# Malaika — Fine-Tune Gemma 4 with Unsloth (Kaggle Only)

**This notebook targets the Unsloth $10K prize.**

Pipeline: ICBHI Audio → mel-spectrogram PNG → Gemma 4 vision fine-tuning via Unsloth QLoRA

**Run on Kaggle with T4 GPU.** Add "respiratory-sound-database" by vbookshelf as dataset input.

In [ ]:
# Cell 1: Install Unsloth (Kaggle-compatible)
!pip install -q unsloth
!pip install -q --no-deps trl datasets librosa soundfile Pillow

In [ ]:
# Cell 2: Authenticate
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
print("Authenticated")

In [ ]:
# Cell 3: Imports + GPU check
import json, os, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import librosa
from PIL import Image

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Load ICBHI dataset (Kaggle dataset input)
ICBHI_PATH = Path("/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database")

if ICBHI_PATH.exists():
    audio_dir = ICBHI_PATH / "audio_and_txt_files"
    audio_files = list(audio_dir.glob("*.wav"))
    print(f"ICBHI dataset: {len(audio_files)} audio files")
else:
    print("ERROR: Add 'respiratory-sound-database' by vbookshelf via Add Data button")
    audio_files = []

In [ ]:
# Cell 5: Parse annotations
def parse_icbhi_annotations(audio_dir):
    records = []
    for txt_file in sorted(audio_dir.glob("*.txt")):
        wav_file = txt_file.with_suffix(".wav")
        if not wav_file.exists(): continue
        parts = txt_file.stem.split("_")
        patient_id = parts[0] if parts else "unknown"
        has_crackle, has_wheeze = False, False
        with open(txt_file) as f:
            for line in f:
                fields = line.strip().split()
                if len(fields) >= 4:
                    if int(fields[2]) == 1: has_crackle = True
                    if int(fields[3]) == 1: has_wheeze = True
        if has_crackle and has_wheeze: label = "both"
        elif has_crackle: label = "crackle"
        elif has_wheeze: label = "wheeze"
        else: label = "normal"
        records.append({"audio_path": str(wav_file), "patient_id": patient_id,
            "label": label, "has_crackle": has_crackle, "has_wheeze": has_wheeze})
    return records

if audio_files:
    records = parse_icbhi_annotations(audio_dir)
    print(f"Parsed {len(records)} recordings:")
    for label, count in sorted(Counter(r["label"] for r in records).items()):
        print(f"  {label}: {count}")

In [ ]:
# Cell 6: Generate spectrograms
SPEC_DIR = Path("/kaggle/working/spectrograms")
SPEC_DIR.mkdir(exist_ok=True)

SPEC_W, SPEC_H = 512, 256

def audio_to_spec(audio_path, output_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        if len(y) == 0: return False
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=50, fmax=4000)
        db = librosa.power_to_db(mel, ref=np.max)
        mn, mx = db.min(), db.max()
        norm = ((db - mn) / (mx - mn) * 255).astype(np.uint8) if mx > mn else np.zeros_like(db, dtype=np.uint8)
        norm = np.flip(norm, axis=0)
        Image.fromarray(norm, mode="L").resize((SPEC_W, SPEC_H), Image.Resampling.LANCZOS).convert("RGB").save(str(output_path))
        return True
    except Exception as e:
        print(f"  Skip {Path(audio_path).name}: {e}")
        return False

if audio_files:
    print("Generating spectrograms...")
    spec_records = []
    for i, r in enumerate(records):
        sp = SPEC_DIR / f"{Path(r['audio_path']).stem}_spec.png"
        if audio_to_spec(r['audio_path'], sp):
            r["spectrogram_path"] = str(sp)
            spec_records.append(r)
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(records)}")
    print(f"Generated {len(spec_records)}/{len(records)} spectrograms")
    records = spec_records

In [ ]:
# Cell 7: Create instruction pairs + split
def describe(r):
    if r["label"] == "normal": return "Normal breath sounds. No adventitious sounds."
    parts = []
    if r["has_wheeze"]: parts.append("Wheezing — continuous bright bands 200-1000 Hz")
    if r["has_crackle"]: parts.append("Crackles — discontinuous vertical bright spots")
    return ". ".join(parts) + "."

INSTRUCTION = ("This is a mel-spectrogram of a child's breathing audio.\n"
    "Vertical: frequency (50-4000 Hz). Horizontal: time. Brightness: intensity.\n"
    "Classify the breath sounds as JSON: "
    '{"wheeze": bool, "stridor": bool, "grunting": bool, "crackles": bool, '
    '"normal": bool, "confidence": float, "description": "..."}')

SYSTEM_MSG = ("You are Malaika, a medical spectrogram analysis assistant. "
    "Classify breath sounds from spectrograms. Respond ONLY with JSON.")

if audio_files:
    pairs = [{"spectrogram_path": r["spectrogram_path"], "label": r["label"],
        "instruction": INSTRUCTION,
        "response": json.dumps({"wheeze": r["has_wheeze"], "stridor": False, "grunting": False,
            "crackles": r["has_crackle"], "normal": r["label"]=="normal",
            "confidence": 0.9, "description": describe(r)})} for r in records]

    pids = list(set(r["patient_id"] for r in records))
    random.seed(42); random.shuffle(pids)
    train_pats = set(pids[:int(len(pids)*0.8)])
    train_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] in train_pats]
    test_pairs = [p for p, r in zip(pairs, records) if r["patient_id"] not in train_pats]
    print(f"Train: {len(train_pairs)} | Test: {len(test_pairs)}")

In [ ]:
# Cell 8: Load model with Unsloth
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    load_in_4bit=True,
)

print(f"Model loaded | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 9: Add LoRA adapter via Unsloth
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 10: Format training data for Unsloth SFTTrainer
from datasets import Dataset

def format_for_training(pairs):
    formatted = []
    for p in pairs:
        formatted.append({"messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_MSG}]},
            {"role": "user", "content": [
                {"type": "image", "image": p["spectrogram_path"]},
                {"type": "text", "text": p["instruction"]},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": p["response"]}]},
        ]})
    return formatted

if audio_files:
    train_formatted = format_for_training(train_pairs)
    train_dataset = Dataset.from_list(train_formatted)
    print(f"Training dataset: {len(train_dataset)} examples")
else:
    # Dummy fallback
    dummy_path = SPEC_DIR / "dummy.png"
    if not dummy_path.exists():
        Image.fromarray(np.random.randint(0, 255, (SPEC_H, SPEC_W, 3), dtype=np.uint8)).save(str(dummy_path))
    train_dataset = Dataset.from_list([{"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MSG}]},
        {"role": "user", "content": [
            {"type": "image", "image": str(dummy_path)},
            {"type": "text", "text": "Classify breath sounds."}]},
        {"role": "assistant", "content": [{"type": "text", "text":
            '{"wheeze": false, "crackles": false, "normal": true, "confidence": 0.9}'}]},
    ]}] * 10)
    print("Using dummy data")

In [ ]:
# Cell 11: Train with Unsloth SFTTrainer
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=2e-4,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="/kaggle/working/breath_sound_lora",
        seed=42,
        report_to="none",
    ),
)

print("Starting Unsloth training...")
t0 = time.monotonic()
result = trainer.train()
train_time = time.monotonic() - t0
print(f"Training complete in {train_time/60:.1f} min")
print(f"  Loss: {result.training_loss:.4f} | Steps: {result.global_step}")

In [ ]:
# Cell 12: Save adapter
ADAPTER_NAME = "/kaggle/working/malaika-breath-sounds-unsloth-lora"

model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Adapter saved to {ADAPTER_NAME}/")

for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Cell 13: Evaluate on test set
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")

print("=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

if audio_files and test_pairs:
    correct, total_test = 0, 0
    for pair in test_pairs[:20]:
        spec_img = Image.open(pair["spectrogram_path"]).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": pair["instruction"]}]}]
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=input_text, images=[spec_img], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            m = re.search(r'\{[^{}]*\}', pred_text)
            if m:
                pred = json.loads(m.group(0))
                exp = json.loads(pair["response"])
                if pred.get("wheeze") == exp.get("wheeze") and pred.get("crackles") == exp.get("crackles"):
                    correct += 1; status = "PASS"
                else: status = "FAIL"
            else: status = "FAIL (no JSON)"
        except: status = "FAIL (parse)"
        total_test += 1
        print(f"  {status} [{pair['label']}] -- {pred_text[:80]}")
    print(f"\nAccuracy: {correct}/{total_test} ({100*correct/total_test:.0f}%)")
else:
    print("No test data")

In [ ]:
# Cell 14: Summary
print("=" * 60)
print("UNSLOTH FINE-TUNING SUMMARY")
print("=" * 60)
print(f"Model:          google/gemma-4-E4B-it")
print(f"Method:         Unsloth QLoRA, r=8, vision+language")
print(f"Dataset:        ICBHI 2017 ({len(records) if audio_files else 0} spectrograms)")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"Training time:  {train_time/60:.1f} min")
print(f"Final loss:     {result.training_loss:.4f}")
print(f"Adapter:        {ADAPTER_NAME}")
print()
print("Innovation: Audio -> mel-spectrogram -> Gemma 4 vision fine-tuning")
print("Bypasses native audio limitation using working vision modality")
print("=" * 60)